# UC Berkeley Capstone Project -- Data Analysis & Modelling

This is a notebook for my UC Berkeley Capstone project.

I have a harvester script running in GCP that collects YouTube video metadata and metrics and stores that in a BQ table. 
Because the script is running in an ongoing basis, and I have limited time before I need to turn in the project, I am going to start working with synthetic data integrated with my real data, and gradually shift over to all real data.

- This will allow me start modelling sooner. 
- The synthetic data will be identifiable in the DataFrame.
- I will be storing a Parquet snapshot of the BQ data extract + model version in a GCS Bucket, so that I can always see which specific data + model produced which specific results.

The workflow here is:

1. Create a snapshot of the BQ data 
1. Add the synthetic data rows
1. Store the dataset in a versioned GCS bucket
1. (when modelling) store model vesrion data + stats in a versioned GCS bucket as well.

TODO (jelani):

1. make the snapshotting proces easier to manage with a global flag for on/off and constants for the different model version names, video snapshog version names, etc. The on/off can live in each code blcok where it is needed so that I'm not hunting through code to update it. 
1. Have this all live in a cell at the top of the notebook to make it easy to access and change.
1. Add the newer cut of real data
1. Make the label for synth data split easier to figure out (and set some math to do the split)
1. Add RnadomForrest


In [ ]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import utils.snapshot_data as snapshot
from utils.snapshot_model import (
    TrainingData, ModelResult, ModelConfig,
    save_model, list_models, compare_models,
)
from utils.snapshot_hyperparameters import save_hyperparams, load_hyperparams, list_hyperparams
from pipeline.run_config import RunConfig as SnapshotConfig
#from pipeline.run_config import DEFAULT_LR_PARAMS_, DEFAULT_RF_PARAMS_, DEFAULT_XGB_PARAMS_

from utils.tune_hyperparameters import tune_model, get_default_param_grid

from data_processing.data_cleanup import build_clean_dataset
from data_processing.feature_engineering import engineer_features
from data_processing.synthetic_data import generate_synthetic_data, combine_real_and_synthetic

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    RocCurveDisplay,
)
from xgboost import XGBClassifier

In [ ]:
RANDOM_SEED = 42

# v3.1 was 80% real + 20% synthetic. This run pulls fresh 100% real data,
# so synthetic generation is off.
USE_SYNTHETIC = False

# =============================================================================
# SNAPSHOT CONFIG
# Controls which snapshots are written to GCS this run, and what version each
# entity gets. Version numbers are read from / written back to GCS at
# config/versions.json, which tracks three independent counters:
#   data.{major,minor} | model.{major,minor} | hyperparams.{major,minor}
#
# Presets (minor bumps by default):
#   "dry_run"             — no snapshots; reads current versions
#   "model_tuning"        — tune + models + hyperparams
#                           → model minor, hyperparams minor
#   "feature_engineering" — final + tune + models + hyperparams
#                           → data minor, model minor, hyperparams minor
#   "new_raw_data"        — raw + final + tune + models_new_data + hyperparams
#                           → data minor, model MAJOR, hyperparams minor
#
# Or chain methods explicitly:
#   config = SnapshotConfig.load().snapshot_final().snapshot_models().build()
#
# Upgrade minor → major explicitly by chaining one of:
#   .snapshot_schema_change()        (data major — columns changed)
#   .snapshot_models_new_data()      (model major — retrained on new data)
#   .snapshot_hyperparams_new_grid() (hyperparams major — new param added)
#
# .use_data_version("3.1") pins raw_version + final_version to the existing
# v3.1 data snapshot; model/hyperparam versions still bump independently.
#
# Hit Restart & Run All — every downstream cell guards on the config flags
# and will either run its work or print a skip message. No manual cell
# skipping required.
# =============================================================================

# Current run: pull a fresh 100% real dataset from BQ (same schema), retrain
# all models on it, reuse the v1.0 hyperparams.
#   data        v3.1 → v3.2   (minor — new pull, same schema)
#   final_suffix → "100real"  (was "mixed_80real"; no longer mixed)
#   model       v3.2 → v4.0   (MAJOR — trained on new data)
#   hyperparams pinned v1.0   (no re-tuning this run)
config = (
    SnapshotConfig.load()
    .snapshot_raw()
    .snapshot_final(suffix="100real")
    .snapshot_models_new_data()
    .use_hyperparam_version("1.0")
    .build()
)

TIER_ORDER     = ['S', 'M', 'L']
VERTICAL_ORDER = ['Education', 'Lifestyle', 'Tech']

In [ ]:
def create_new_snapshot(
    version_name: str, additional_notes: str = ""
):
    """
    Creates snapshots in GCS bucket for:
      - Channel baseline data
      - Channel median data
      - Video snapshot data

    Snapshots can be viewed via `snapshot.list_snapshots` and loaded via
    `snapshot.load_videos` and `snapshot.load_baselines`

    version_name: Name like 'v1.0_real'
    additional_notes: Descriptive notes. `version_name` is automatically added.

    Returns: (video_df, baseline_df, median_df, video_meta, baseline_meta)
    """
    video_df, video_meta = snapshot.snapshot_video_data(version_tag=version_name, 
            notes=f"Video data version {version_name}. {additional_notes}",
    )

    baseline_df, median_df, baseline_meta = snapshot.snapshot_baselines(
        version_tag=version_name,
        notes=f"Channel baselines corresponding to {version_name} video snapshot. {additional_notes}",
    )
    return (video_df, baseline_df, median_df, video_meta, baseline_meta)

In [ ]:
if config.take_snapshot_raw:
    additional_notes = "14.5k 7d upload triplets — first 100% real data snapshot."
    video_df, baseline_df, median_df, video_meta, baseline_meta = create_new_snapshot(
        version_name=config.raw_version, additional_notes=additional_notes
    )

# See what's available
snapshot.list_snapshots()

In [ ]:
# Raw BQ data is only loaded when taking a new snapshot.
# For model-tuning / reuse runs, df_combined is loaded directly from the
# final-dataset snapshot below — skip raw loading and the cleaning/EDA cells.

if config.take_snapshot_raw or config.take_snapshot_final:
    df_videos, meta = snapshot.load_videos(config.raw_version)
    df_baselines, df_medians, baseline_meta = snapshot.load_baselines(config.raw_version)

    print(f"Video snapshots: {len(df_videos)}")
    print(f"Baseline videos: {len(df_baselines)}")
    print(f"Baseline medians: {len(df_medians)}")

    df_videos.head()
else:
    df_videos = df_baselines = df_medians = None
    print(f"Model-tuning mode — raw load skipped. Using {config.final_version} directly.")

In [ ]:
if df_baselines is None:
    print(f"[skip] raw-data EDA — model-tuning mode (reusing {config.final_version})")
else:
    df_baselines.head()

In [ ]:
if df_medians is None:
    print(f"[skip] raw-data EDA — model-tuning mode (reusing {config.final_version})")
else:
    df_medians.head()

In [ ]:
if df_videos is None:
    print(f"[skip] raw-data EDA — model-tuning mode (reusing {config.final_version})")
else:
    print(df_videos.describe())
    print(df_videos.info())

In [ ]:
# Initial cleaning + feature engineering — runs only when raw data was loaded.
# In model-tuning mode, df_model is reconstructed from the final-dataset
# snapshot in the save/load cell below.
if df_videos is None:
    df_model = None
    print(f"[skip] feature engineering — will reconstruct df_model from {config.final_version}")
else:
    # NULL contains_synthetic_media means the video couldn't be assessed
    # (e.g. private or deleted) — exclude those rows.
    df_videos.dropna(subset=["contains_synthetic_media"], inplace=True)

    df_clean = build_clean_dataset(df_videos, df_medians)
    df_model = engineer_features(df_clean)

    print(df_model.head())

In [ ]:
def get_synth_rows_proportion(num_real_rows: int, target_real_pct: float):
    return math.floor(float(num_real_rows) / target_real_pct) - num_real_rows

In [ ]:
if config.take_snapshot_final and USE_SYNTHETIC:
    target_synth_rows = get_synth_rows_proportion(num_real_rows=len(df_model), target_real_pct=0.8)
    print(f"target_synth_rows: {target_synth_rows}")
    df_synth = generate_synthetic_data(df_model, num_rows=target_synth_rows, seed=RANDOM_SEED)
    df_combined = combine_real_and_synthetic(df_model, df_synth)
elif config.take_snapshot_final and not USE_SYNTHETIC:
    target_synth_rows = 0
    df_combined = df_model.copy()
    df_combined['contains_synthetic_data'] = False
    print(f"Synthetic data disabled — {len(df_combined)} real rows, 0 synthetic")
else:
    target_synth_rows = 0
    df_combined = None
    print(f"[skip] synthetic generation — will load {config.final_version} directly in next cell.")

In [ ]:
if config.take_snapshot_final:
    if USE_SYNTHETIC:
        notes = f"{len(df_model)} real + {target_synth_rows} synthetic (GaussianCopula seed={RANDOM_SEED}), real channel baselines"
    else:
        notes = f"{len(df_model)} real rows, 0 synthetic (100% real data)"
    snapshot_meta = snapshot.save_snapshot(
        df=df_combined, version_tag=config.final_version, notes=notes
    )

# Load so we aren't creating a snapshot every time the notebook runs.
# The final-dataset snapshot is the shared artifact across all run paths;
# downstream cells expect df_combined to be defined regardless of which flags were set.
try:
    df_combined, _ = snapshot.load_videos(config.final_version)
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"No final-dataset snapshot found at '{config.final_version}'. "
        f"Either chain .snapshot_final() in the config cell to create one, "
        f"or pin to an existing version via .use_data_version(...). "
        f"Original error: {e}"
    ) from e

# In model-tuning mode df_model was never built from raw data — reconstruct
# it from the final-dataset snapshot by filtering out synthetic rows.
if df_videos is None:
    df_model = df_combined[~df_combined["contains_synthetic_data"]].copy()
    print(f"Reconstructed df_model from final-dataset snapshot: {len(df_model)} real rows")

df_train = df_combined

# Never evaluate on synthetic data
df_eval = df_combined[~df_combined["contains_synthetic_data"]]

df_train.sample(5)

In [ ]:
# --- Define feature columns (exclude IDs, text, timestamps, target, metadata) ---

# Add 7d metric columns to the exclusion list
EXCLUDE_7D = [c for c in df_combined.columns if c.endswith('_7d')]
print(f"Excluding 7d features: {EXCLUDE_7D}")

EXCLUDE_COLS = [
    # IDs and text
    'video_id', 'channel_id', 'channel_handle', 'title', 'description', 'tags',
    'category_id', 'category_name',
    # Timestamps
    'published_at', 'poll_timestamp_upload', 'poll_timestamp_24h', 'poll_timestamp_7d'
    # Target and intermediate
    'above_baseline', 'engagement_7d', 'baseline_engagement',
    # Metadata flags
    'contains_synthetic_data', 'contains_synthetic_media',
    # Baseline raw (already captured in baseline_engagement)
    'baseline_channel_handle', 'baseline_video_count',
    'baseline_median_views', 'baseline_median_likes',
    'baseline_median_comments', 'baseline_median_engagement_rate',
    # Categorical groupings (use the encoded versions instead)
    'vertical', 'tier',
    # Redundant with duration_seconds
    'duration_minutes',
    # Part of the data harvesting process, not useful for modelling
    'hours_since_publish_upload', 'hours_since_publish_24h',
    # Superseded by has_face (binary flag is more reliable signal)
    'face_count',
]

EXCLUDE_COLS.extend(EXCLUDE_7D)

feature_cols = [c for c in df_combined.columns if c not in EXCLUDE_COLS]
print(f"\nFeature columns ({len(feature_cols)}):")
print(feature_cols)

## Exploratory Data Analysis

Real data only (synthetic rows excluded). Run after feature engineering to reflect engineered column names.

In [ ]:
import seaborn as sns

# Real data only for all EDA plots.
# Use df_model (always freshly feature-engineered) rather than df_combined,
# which is loaded from a GCS snapshot and may not have the latest engineered columns.
_df_eda = df_model.copy()

# Sanity check — catch stale df_model before hitting a confusing KeyError mid-plot
_expected_cols = [
    'view_velocity_ratio', 'view_count_velocity_24h', 'view_velocity_acceleration',
    'like_rate_24h', 'view_count_upload_vs_baseline', 'has_face', 'duration_bucket',
]
_missing = [c for c in _expected_cols if c not in _df_eda.columns]
if _missing:
    raise RuntimeError(
        f"df_model is missing engineered columns: {_missing}\n"
        "Re-run the cleaning + feature engineering cells (build_clean_dataset / engineer_features) first."
    )


# =============================================================================
# Plot 1: Above-baseline label rate by vertical and by tier
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, order in [
    (axes[0], 'vertical', VERTICAL_ORDER),
    (axes[1], 'tier',     TIER_ORDER),
]:
    stats = (
        _df_eda.groupby(col)['above_baseline']
        .agg(rate='mean', n='count')
        .reindex(order)
    )
    bars = ax.bar(stats.index, stats['rate'] * 100, color='steelblue', edgecolor='white')
    for bar, (_, row) in zip(bars, stats.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"n={int(row['n'])}",
            ha='center', va='bottom', fontsize=9,
        )
    ax.axhline(50, ls='--', color='gray', alpha=0.6, label='50% reference')
    ax.set_ylabel('Above Baseline (%)')
    ax.set_ylim(0, 85)
    ax.set_title(f'Label Rate by {col.capitalize()}')

plt.suptitle('Target Label Rate — Real Data Only', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


# =============================================================================
# Plot 2: Engagement rate distribution by tier × vertical (box plot)
# Answers: do higher-sub channels (tier) have higher engagement, and does
# that pattern hold consistently across verticals?
# =============================================================================

fig, ax = plt.subplots(figsize=(11, 5))

sns.boxplot(
    data=_df_eda,
    x='tier', y='engagement_7d',
    hue='vertical',
    order=TIER_ORDER,
    hue_order=VERTICAL_ORDER,
    palette='Set2',
    width=0.6,
    flierprops=dict(marker='o', markersize=3, alpha=0.4),
    ax=ax,
)

ax.set_xlabel('Tier (S = smallest, L = largest)')
ax.set_ylabel('Engagement Rate (7d)')
ax.set_title('Engagement Rate Distribution by Tier and Vertical')
ax.set_ylim(0, 0.3)
ax.legend(title='Vertical', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()


# =============================================================================
# Plot 3: Heatmap — label rate and sample size per vertical × tier
# =============================================================================

pivot_rate = (
    _df_eda.groupby(['vertical', 'tier'])['above_baseline']
    .mean()
    .unstack('tier')
    .reindex(index=VERTICAL_ORDER, columns=TIER_ORDER)
    * 100
)
pivot_n = (
    _df_eda.groupby(['vertical', 'tier'])['above_baseline']
    .count()
    .unstack('tier')
    .reindex(index=VERTICAL_ORDER, columns=TIER_ORDER)
)

# Build annotation strings: "XX.X%\nn=YYY"
annot = (
    pivot_rate.round(1).astype(str).add('%\n') +
    'n=' + pivot_n.astype(str)
)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    pivot_rate,
    ax=ax,
    annot=annot,
    fmt='',
    cmap='RdYlGn',
    vmin=35, vmax=65,
    linewidths=0.8,
    cbar_kws={'label': 'Above Baseline (%)'},
)
ax.set_title('Label Rate & Sample Size per Segment\n(vertical × tier, real data only)')
ax.set_xlabel('Tier')
ax.set_ylabel('Vertical')
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Velocity feature distributions — split by label
# Shows whether each velocity metric is actually discriminative.
# Three features tell the story: raw speed, speed vs channel norm, acceleration.
# =============================================================================

_velocity_features = [
    ('view_count_velocity_24h', 'View Velocity 24h\n(views/hour, upload→24h)'),
    ('view_velocity_ratio',     'View Velocity Ratio\n(velocity / channel baseline median)'),
    ('view_velocity_acceleration', 'View Velocity Acceleration\n(24h velocity / upload velocity)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (col, title) in zip(axes, _velocity_features):
    for label, color, name in [(0, '#e05c5c', 'Below Baseline'), (1, '#4c72b0', 'Above Baseline')]:
        vals = _df_eda.loc[_df_eda['above_baseline'] == label, col].dropna()
        # Cap at 99th percentile to prevent extreme outliers from flattening the histogram
        cap = vals.quantile(0.99)
        vals = vals.clip(upper=cap)
        ax.hist(vals, bins=40, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Value (capped at 99th pct)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Velocity Feature Distributions by Label — Real Data Only', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Correlation matrix — mixed (train) data vs real-only data
# Comparing the two shows whether synthetic data is distorting feature relationships.
# =============================================================================

_df_mixed = df_combined[feature_cols + ['above_baseline']].copy()
_df_real  = df_combined[~df_combined['contains_synthetic_data']][feature_cols + ['above_baseline']].copy()

_n_mixed   = len(_df_mixed)
_n_real    = len(_df_real)
_n_synth   = _n_mixed - _n_real
_synth_pct = round(_n_synth / _n_mixed * 100, 1)
_real_pct  = round(_n_real  / _n_mixed * 100, 1)

fig, axes = plt.subplots(1, 2, figsize=(28, 12))

for ax, df_plot, title in [
    (
        axes[0],
        _df_mixed,
        f'Correlation Matrix — Mixed Training Data\n'
        f'n={_n_mixed} ({_real_pct}% real + {_synth_pct}% synthetic)',
    ),
    (
        axes[1],
        _df_real,
        f'Correlation Matrix — Real Data Only\n'
        f'n={_n_real} (100% real)',
    ),
]:
    corr = df_plot.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle only
    sns.heatmap(
        corr,
        ax=ax,
        mask=mask,
        cmap='coolwarm',
        vmin=-1, vmax=1,
        center=0,
        linewidths=0.3,
        annot=False,
        cbar_kws={'shrink': 0.6},
    )
    ax.set_title(title, fontsize=11, pad=10)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle(
    'Feature Correlation Comparison: Mixed vs Real-Only\n'
    '(large differences between panels suggest synthetic data is altering feature relationships)',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Scatter plots — do pairs of features separate the classes?
#
# Plot A: view_velocity_ratio vs like_rate_24h
#   Speed relative to channel norm (x) vs engagement quality (y).
#   Two independent signals — separation here isn't just a tier effect.
#
# Plot B: view_count_upload_vs_baseline vs view_velocity_acceleration
#   Early traction vs channel norm (x) vs momentum trajectory (y).
#   Tests whether "starting hot and accelerating" predicts above-baseline.
# =============================================================================

_LABEL_COLORS = {0: '#e05c5c', 1: '#4c72b0'}
_LABEL_NAMES  = {0: 'Below Baseline', 1: 'Above Baseline'}

_scatter_pairs = [
    (
        'view_velocity_ratio',
        'like_rate_24h',
        'View Velocity Ratio\n(velocity / channel baseline median)',
        'Like Rate at 24h\n(likes / views)',
        'Plot A: Velocity vs Engagement Quality',
    ),
    (
        'view_count_upload_vs_baseline',
        'view_velocity_acceleration',
        'View Count at Upload vs Baseline Median\n(early traction)',
        'View Velocity Acceleration\n(24h velocity / upload velocity)',
        'Plot B: Early Traction vs Momentum Trajectory',
    ),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (xcol, ycol, xlabel, ylabel, title) in zip(axes, _scatter_pairs):
    for label in [0, 1]:
        sub = _df_eda[_df_eda['above_baseline'] == label][[xcol, ycol]].dropna()

        # Cap both axes at 99th percentile so outliers don't collapse the main cloud
        xc = sub[xcol].clip(upper=sub[xcol].quantile(0.99))
        yc = sub[ycol].clip(upper=sub[ycol].quantile(0.99))

        ax.scatter(
            xc, yc,
            color=_LABEL_COLORS[label],
            alpha=0.25, s=12,
            label=_LABEL_NAMES[label],
        )

        # Density contours to show where each class concentrates
        try:
            sns.kdeplot(
                x=xc, y=yc,
                ax=ax,
                color=_LABEL_COLORS[label],
                levels=4,
                linewidths=1.2,
                alpha=0.7,
            )
        except Exception:
            pass  # KDE can fail on degenerate distributions; dots still show

    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle(
    'Feature Pair Scatter Plots — Real Data Only\n'
    '(contours show class density; axes capped at 99th percentile)',
    fontsize=11, y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# === Split: real data only for evaluation ===

df_real = df_combined[~df_combined['contains_synthetic_data']].copy()
df_synth_rows = df_combined[df_combined['contains_synthetic_data']].copy()

print(f"Real rows:      {len(df_real)}")
print(f"Synthetic rows: {len(df_synth_rows)}")

# Split real data into train/test (stratified on target)
X_real = df_real[feature_cols]
y_real = df_real['above_baseline']

X_real_train, X_test, y_real_train, y_test = train_test_split(
    X_real, y_real,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y_real,
)

# Add synthetic rows to training set only
X_synth = df_synth_rows[feature_cols]
y_synth = df_synth_rows['above_baseline']

X_train = pd.concat([X_real_train, X_synth], ignore_index=True)
y_train = pd.concat([y_real_train, y_synth], ignore_index=True)

# Handle NaN/inf
X_train = X_train.fillna(0).replace([np.inf, -np.inf], 0)
X_test = X_test.fillna(0).replace([np.inf, -np.inf], 0)

print(f"\nTraining set: {len(X_train)} rows "
      f"({len(X_real_train)} real + {len(X_synth)} synthetic)")
print(f"Test set:     {len(X_test)} rows (100% real)")
print(f"\nTarget balance (train): {y_train.value_counts().to_dict()}")
print(f"Target balance (test):  {y_test.value_counts().to_dict()}")

# Scale for LR (RF and XGBoost use unscaled)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Hyperparameter Tuning

Run each block below to search for best params per model. Results are applied to the model instances before training in the cells that follow.

`search_strategy` options: `"random"` (default, fast), `"halving"` (good middle ground), `"grid"` (exhaustive, slow).

In [ ]:
from pipeline.run_config import DEFAULT_LR_PARAMS_, DEFAULT_RF_PARAMS_, DEFAULT_XGB_PARAMS_

def param_grid_(model):
    """Use config.new_grids override if provided, otherwise use the default grid."""
    return config.new_grids.get(type(model).__name__) or get_default_param_grid(model)

if config.tune_models:
    # --- Logistic Regression ---
    lr_base_ = LogisticRegression(random_state=RANDOM_SEED)
    lr_result_ = tune_model(
        lr_base_, X_train_scaled, y_train,
        param_grid=param_grid_(lr_base_),
        search_strategy=config.search_strategy,
        n_iter=config.search_n_iter,
        cv=config.search_cv,
        scoring=config.search_scoring,
        random_state=RANDOM_SEED,
    )
    LR_PARAMS = lr_result_['best_params']

    # --- Random Forest ---
    rf_base_ = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)
    rf_result_ = tune_model(
        rf_base_, X_train, y_train,
        param_grid=param_grid_(rf_base_),
        search_strategy=config.search_strategy,
        n_iter=config.search_n_iter,
        cv=config.search_cv,
        scoring=config.search_scoring,
        random_state=RANDOM_SEED,
    )
    RF_PARAMS = rf_result_['best_params']

    # --- XGBoost ---
    xgb_base_ = XGBClassifier(random_state=RANDOM_SEED, n_jobs=-1, eval_metric='logloss')
    xgb_result_ = tune_model(
        xgb_base_, X_train, y_train,
        param_grid=param_grid_(xgb_base_),
        search_strategy=config.search_strategy,
        n_iter=config.search_n_iter,
        cv=config.search_cv,
        scoring=config.search_scoring,
        random_state=RANDOM_SEED,
    )
    XGB_PARAMS = xgb_result_['best_params']

    print("\n=== Best params summary ===")
    print(f"LR:  {LR_PARAMS}")
    print(f"RF:  {RF_PARAMS}")
    print(f"XGB: {XGB_PARAMS}")

else:
    # Load best params from the GCS hyperparam snapshot for this version.
    # Falls back to defaults (from snapshot_config) if no snapshot exists yet.
    try:
        stored_ = load_hyperparams(config.hyperparam_version)
        LR_PARAMS  = stored_['params']['LogisticRegression']
        RF_PARAMS  = stored_['params']['RandomForestClassifier']
        XGB_PARAMS = stored_['params']['XGBClassifier']
        print(f"Loaded hyperparams from GCS: {config.hyperparam_version}")
    except FileNotFoundError:
        LR_PARAMS, RF_PARAMS, XGB_PARAMS = DEFAULT_LR_PARAMS_, DEFAULT_RF_PARAMS_, DEFAULT_XGB_PARAMS_
        print(f"No hyperparam snapshot for {config.hyperparam_version} — using defaults.")
        print("Run with config.tune() to generate and snapshot tuned params.")

In [ ]:
# === Model 1: Logistic Regression with L1 (LASSO) ===

model_lr = LogisticRegression(random_state=42, **LR_PARAMS)
model_lr.fit(X_train_scaled, y_train)

# Predictions on real-only test set
y_pred = model_lr.predict(X_test_scaled)
y_pred_proba = model_lr.predict_proba(X_test_scaled)[:, 1]

print("=" * 60)
print("Logistic Regression (L1) — Evaluated on Real Data Only")
print("=" * 60)
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Below Baseline', 'Above Baseline'])}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

In [ ]:
# === Feature Importance (L1 coefficients) ===

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': model_lr.coef_[0],
    'abs_coefficient': np.abs(model_lr.coef_[0]),
}).sort_values('abs_coefficient', ascending=False)

# Features that L1 zeroed out (not useful)
zeroed = coef_df[coef_df['coefficient'] == 0]
print(f"Features zeroed out by L1: {len(zeroed)}/{len(coef_df)}")

# Top 20 most important
print("\nTop 20 Features:")
print(coef_df.head(20).to_string(index=False))

In [ ]:
# === Visualizations ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0])
axes[0].set_title('ROC Curve (Real Test Data)')

# Top 15 feature coefficients
top_15 = coef_df.head(15)
colors = ['green' if c > 0 else 'red' for c in top_15['coefficient']]
axes[1].barh(top_15['feature'], top_15['coefficient'], color=colors)
axes[1].set_xlabel('Coefficient')
axes[1].set_title('Top 15 Features (L1 Logistic Regression)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# === Model 2: Random Forest (scale-invariant, no scaling needed) ===

model_rf = RandomForestClassifier(random_state=42, n_jobs=-1, **RF_PARAMS)
model_rf.fit(X_train, y_train)

# Predictions on real-only test set
y_pred_rf = model_rf.predict(X_test)
y_pred_proba_rf = model_rf.predict_proba(X_test)[:, 1]

print("=" * 60)
print("Random Forest — Evaluated on Real Data Only")
print("=" * 60)
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")
print(f"\n{classification_report(y_test, y_pred_rf, target_names=['Below Baseline', 'Above Baseline'])}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_rf)}")

In [ ]:
# === RF Feature Importance ===

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_rf.feature_importances_,
}).sort_values('importance', ascending=False)

print("Top 20 Features (Random Forest):")
print(importance_df.head(20).to_string(index=False))

In [ ]:
# === Side-by-side comparison ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves compared
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], name='LR (L1)')
RocCurveDisplay.from_predictions(y_test, y_pred_proba_rf, ax=axes[0], name='Random Forest')
axes[0].set_title('ROC Comparison (Real Test Data)')

# RF top 15 feature importances
top_15_rf = importance_df.head(15)
axes[1].barh(top_15_rf['feature'], top_15_rf['importance'], color='steelblue')
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 15 Features (Random Forest)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# === Model 3: XGBoost ===

model_xgb = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', **XGB_PARAMS)
model_xgb.fit(X_train, y_train)

# Predictions on real-only test set
y_pred_xgb = model_xgb.predict(X_test)
y_pred_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

print("=" * 60)
print("XGBoost — Evaluated on Real Data Only")
print("=" * 60)
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")
print(f"\n{classification_report(y_test, y_pred_xgb, target_names=['Below Baseline', 'Above Baseline'])}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_xgb)}")

In [ ]:
# === XGBoost Feature Importance ===

importance_df_xgb = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_xgb.feature_importances_,
}).sort_values('importance', ascending=False)

print("Top 20 Features (XGBoost):")
print(importance_df_xgb.head(20).to_string(index=False))

In [ ]:
# === All three models compared ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], name='LR (L1)')
RocCurveDisplay.from_predictions(y_test, y_pred_proba_rf, ax=axes[0], name='Random Forest')
RocCurveDisplay.from_predictions(y_test, y_pred_proba_xgb, ax=axes[0], name='XGBoost')
axes[0].set_title('ROC Comparison (Real Test Data)')

top_15_xgb = importance_df_xgb.head(15)
axes[1].barh(top_15_xgb['feature'], top_15_xgb['importance'], color='darkorange')
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 15 Features (XGBoost)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# === Model comparison table: precision, recall, F1, support, accuracy, AUC ===

def _model_report(name, y_true, y_pred, y_pred_proba):
    r = classification_report(
        y_true, y_pred,
        target_names=['Below Baseline', 'Above Baseline'],
        output_dict=True,
    )
    return {
        ('Below Baseline', 'Precision'): round(r['Below Baseline']['precision'], 3),
        ('Below Baseline', 'Recall'):    round(r['Below Baseline']['recall'], 3),
        ('Below Baseline', 'F1'):        round(r['Below Baseline']['f1-score'], 3),
        ('Below Baseline', 'Support'):   int(r['Below Baseline']['support']),
        ('Above Baseline', 'Precision'): round(r['Above Baseline']['precision'], 3),
        ('Above Baseline', 'Recall'):    round(r['Above Baseline']['recall'], 3),
        ('Above Baseline', 'F1'):        round(r['Above Baseline']['f1-score'], 3),
        ('Above Baseline', 'Support'):   int(r['Above Baseline']['support']),
        ('Overall', 'Accuracy'):         round(r['accuracy'], 3),
        ('Overall', 'ROC-AUC'):          round(roc_auc_score(y_true, y_pred_proba), 3),
    }

rows = {
    'LR (L1)':        _model_report('LR (L1)',       y_test, y_pred,     y_pred_proba),
    'Random Forest':  _model_report('Random Forest', y_test, y_pred_rf,  y_pred_proba_rf),
    'XGBoost':        _model_report('XGBoost',       y_test, y_pred_xgb, y_pred_proba_xgb),
}

df_comparison = pd.DataFrame(rows).T
df_comparison.index.name = 'Model'
df_comparison.columns = pd.MultiIndex.from_tuples(df_comparison.columns)

# Highlight best value per column (excluding Support, which is fixed)
highlight_cols = [c for c in df_comparison.columns if c[1] != 'Support']
df_comparison.style \
    .highlight_max(subset=highlight_cols, color='#d4edda') \
    .format({c: '{:.3f}' for c in highlight_cols})

## Ensemble Model — Soft Voting (RF + XGB)

LR is excluded: its AUC (~0.77) is too far behind RF (~0.85) and XGB (~0.88) for diversity to offset the weaker probability estimates in the average. Weights `[1, 2]` (RF : XGB) respect XGB's stronger individual performance. Both components are scale-invariant, so no scaler wrapping is needed.

In [ ]:
# === Soft Voting Ensemble (RF + XGB, weighted toward XGB) ===
# LR is excluded: AUC gap of ~0.08-0.11 vs RF/XGB is too wide for diversity to
# offset the weaker probability estimates. Weights [1, 2] respect XGB's stronger
# individual performance. Both components are scale-invariant — no scaler needed.

model_ensemble = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, **RF_PARAMS)),
        ('xgb', XGBClassifier(random_state=RANDOM_SEED, n_jobs=-1, eval_metric='logloss', **XGB_PARAMS)),
    ],
    voting='soft',
    weights=[1, 2],
)
model_ensemble.fit(X_train, y_train)

y_pred_ensemble = model_ensemble.predict(X_test)
y_pred_proba_ensemble = model_ensemble.predict_proba(X_test)[:, 1]

print("=" * 60)
print("Soft Voting Ensemble (RF + XGB, weights=[1, 2]) — Real Test Data")
print("=" * 60)
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba_ensemble):.4f}")
print(f"\n{classification_report(y_test, y_pred_ensemble, target_names=['Below Baseline', 'Above Baseline'])}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_ensemble)}")

In [ ]:
# === Ensemble vs base models: ROC + comparison table ===

fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(y_test, y_pred_proba,          ax=ax, name='LR (L1)')
RocCurveDisplay.from_predictions(y_test, y_pred_proba_rf,       ax=ax, name='Random Forest')
RocCurveDisplay.from_predictions(y_test, y_pred_proba_xgb,      ax=ax, name='XGBoost')
RocCurveDisplay.from_predictions(y_test, y_pred_proba_ensemble, ax=ax, name='Ensemble (RF+XGB, 1:2)')
ax.set_title('ROC Comparison — Ensemble vs Base Models (Real Test Data)')

plt.tight_layout()
plt.show()

# 4-way comparison table (reuses _model_report from the 3-way table above)
rows_with_ensemble = {
    'LR (L1)':        _model_report('LR (L1)',       y_test, y_pred,          y_pred_proba),
    'Random Forest':  _model_report('Random Forest', y_test, y_pred_rf,       y_pred_proba_rf),
    'XGBoost':        _model_report('XGBoost',       y_test, y_pred_xgb,      y_pred_proba_xgb),
    'Ensemble':       _model_report('Ensemble',      y_test, y_pred_ensemble, y_pred_proba_ensemble),
}

df_comparison_ensemble = pd.DataFrame(rows_with_ensemble).T
df_comparison_ensemble.index.name = 'Model'
df_comparison_ensemble.columns = pd.MultiIndex.from_tuples(df_comparison_ensemble.columns)

highlight_cols = [c for c in df_comparison_ensemble.columns if c[1] != 'Support']
df_comparison_ensemble.style \
    .highlight_max(subset=highlight_cols, color='#d4edda') \
    .format({c: '{:.3f}' for c in highlight_cols})

In [ ]:
training_data = TrainingData.from_splits(X_train, y_train, X_test, y_test, X_synth)

if config.take_snapshot_hyperparams:
    save_hyperparams(
        params={
            'LogisticRegression':     LR_PARAMS,
            'RandomForestClassifier': RF_PARAMS,
            'XGBClassifier':          XGB_PARAMS,
            'VotingClassifier':       {'voting': 'soft', 'estimators': ['rf', 'xgb'], 'weights': [1, 2]},
        },
        version_tag=config.hyperparam_version,
        search_config=config.search_config,
        notes=f"Tuned on {config.final_version} training data",
    )

if config.take_snapshot_models:
    result_lr_ = ModelResult.from_sklearn(model_lr, X_test_scaled, y_test, feature_cols)
    model_config_lr_ = ModelConfig.from_model(model_lr)
    save_model(
        model=model_lr,
        scaler=scaler,
        feature_cols=feature_cols,
        version_tag=f"{config.model_version}_lr_l1",
        data_snapshot_tag=config.final_version,
        training_data=training_data,
        result=result_lr_,
        config=model_config_lr_,
        notes="LR L1 baseline, all 7d features excluded",
    )

    result_rf_ = ModelResult.from_sklearn(model_rf, X_test, y_test, feature_cols)
    model_config_rf_ = ModelConfig.from_model(model_rf)
    save_model(
        model=model_rf,
        scaler=None,
        feature_cols=feature_cols,
        version_tag=f"{config.model_version}_rf",
        data_snapshot_tag=config.final_version,
        training_data=training_data,
        result=result_rf_,
        config=model_config_rf_,
        notes="Random Forest, unscaled features",
    )

    result_xgb_ = ModelResult.from_sklearn(model_xgb, X_test, y_test, feature_cols)
    model_config_xgb_ = ModelConfig.from_model(model_xgb)
    save_model(
        model=model_xgb,
        scaler=None,
        feature_cols=feature_cols,
        version_tag=f"{config.model_version}_xgb",
        data_snapshot_tag=config.final_version,
        training_data=training_data,
        result=result_xgb_,
        config=model_config_xgb_,
        notes="XGBoost, unscaled features",
    )

    result_ensemble_ = ModelResult.from_sklearn(model_ensemble, X_test, y_test, feature_cols)
    model_config_ensemble_ = ModelConfig.from_model(model_ensemble)
    save_model(
        model=model_ensemble,
        scaler=None,
        feature_cols=feature_cols,
        version_tag=f"{config.model_version}_ensemble",
        data_snapshot_tag=config.final_version,
        training_data=training_data,
        result=result_ensemble_,
        config=model_config_ensemble_,
        notes="Soft voting ensemble (RF+XGB, weights=[1,2]); LR excluded due to ~0.08-0.11 AUC gap",
    )

# Compare all saved models
list_models()

In [ ]:
# === Cross-version model comparison (all saved snapshots) ===

df_all_models = compare_models()

metric_cols = ['accuracy', 'roc_auc', 'f1_above', 'f1_below', 'precision_above', 'recall_above', 'precision_below', 'recall_below']

df_all_models.style \
    .highlight_max(subset=metric_cols, color='#d4edda') \
    .highlight_min(subset=metric_cols, color='#f8d7da') \
    .format({c: '{:.3f}' for c in metric_cols})

In [ ]:
# === Commit version numbers to GCS after all snapshots succeed ===
# Only writes if any snapshots were active this run.
config.commit()

Train rows: 5298   Test rows: 1010   Total: 6308


train_total  train_above_median  test_total  \
vertical  tier                                                
Education S             381                 202          61   
          M             515                 330          98   
          L             713                 414         129   
Lifestyle S             634                 359         120   
          M             734                 386         128   
          L             495                 300          95   
Tech      S             553                 298         101   
          M             776                 477         157   
          L             497                 266         121   

                test_above_median  total_videos  
vertical  tier                                   
Education S                    32           442  
          M                    57           613  
          L                    77           842  
Lifestyle S                    63           754  
          M                    66           862  
          L                    52           590  
Tech      S                    49           654  
          M                   104           933  
          L                    66           618

In [ ]:
# === Inventory ===
 snapshot.list_snapshots()
 list_models()
 list_hyperparams()

# === Train/test composition by vertical × tier ===
# X_train/X_test don't carry vertical/tier (excluded via EXCLUDE_COLS), so
# reconstruct the split at the df level using the same seed to re-attach
# those columns. Synthetic rows go entirely to train (matches Cell 20 logic).

_real_df  = df_combined[~df_combined['contains_synthetic_data']]
_synth_df = df_combined[df_combined['contains_synthetic_data']]

_train_real_df, _test_df = train_test_split(
    _real_df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=_real_df['above_baseline'],
)
_train_df = pd.concat([_train_real_df, _synth_df])

def _composition_(df, prefix):
    g = df.groupby(['vertical', 'tier'], observed=True).agg(
        total=('above_baseline', 'size'),
        above_median=('above_baseline', 'sum'),
    )
    g.columns = [f'{prefix}_{c}' for c in g.columns]
    return g

_full_index = pd.MultiIndex.from_product(
    [VERTICAL_ORDER, TIER_ORDER], names=['vertical', 'tier'],
)

df_split_composition = (
    pd.concat(
        [_composition_(_train_df, 'train'), _composition_(_test_df, 'test')],
        axis=1,
    )
    .reindex(_full_index, fill_value=0)
    .astype(int)
)
df_split_composition['total_videos'] = (
    df_split_composition['train_total'] + df_split_composition['test_total']
)
df_split_composition = df_split_composition[[
    'train_total', 'train_above_median',
    'test_total',  'test_above_median',
    'total_videos',
]]

print(f"Train rows: {len(_train_df)}   Test rows: {len(_test_df)}   Total: {len(df_combined)}")
df_split_composition
